# Notebook guiado — Árboles de Decisión, Random Forest y XGBoost

**RA2 · Programación de Inteligencia Artificial**  
**Notebook de refuerzo · Google Colab**

En este notebook estudiaremos tres modelos basados en árboles siguiendo una progresión natural:

**Árbol de decisión → Random Forest → XGBoost**

La idea es entender primero cómo un árbol toma decisiones, después cómo muchos árboles pueden combinarse en un bosque y, finalmente, cómo el **boosting** construye árboles de forma secuencial para ir corrigiendo errores.

## ¿Por qué incluimos XGBoost aquí?

Porque conceptualmente encaja muy bien después de Random Forest:

- **Árbol de decisión:** un único árbol.
- **Random Forest:** muchos árboles entrenados con aleatoriedad y combinados.
- **XGBoost:** muchos árboles construidos secuencialmente para mejorar el modelo acumulado.

Así podemos comparar dos grandes ideas de ensembles:

**Bagging** → Random Forest  
**Boosting** → XGBoost

## Recorrido

1. Anatomía de un árbol
2. Cómo se elige una división
3. Impureza Gini
4. Árbol paso a paso
5. Visualización del árbol
6. Fronteras de decisión
7. Profundidad y sobreajuste
8. Regularización del árbol
9. Random Forest
10. Bootstrap y aleatoriedad de variables
11. Votación del bosque
12. Comparación árbol vs. bosque
13. OOB Score
14. Importancia de variables
15. XGBoost y boosting
16. Learning rate y número de árboles
17. Curva de aprendizaje de XGBoost
18. Early stopping
19. Comparación real de los tres modelos
20. ROC, matrices de confusión y métricas
21. Importancia por permutación
22. Escalado: por qué aquí no suele ser necesario
23. Cuándo elegir cada modelo
24. Mini reto final

> El objetivo es comprender las ideas. No necesitas memorizar todos los hiperparámetros.


# 0. Preparar el entorno

`scikit-learn` está disponible habitualmente en Google Colab.

XGBoost es una librería independiente, así que actualizaremos su paquete antes de comenzar.


In [ ]:
!pip -q install -U xgboost


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
import xgboost

print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgboost.__version__)


# 1. ¿Qué es un árbol de decisión?

Un árbol de decisión realiza una secuencia de preguntas.

Ejemplo conceptual:

**¿longitud del pétalo ≤ 2.45?**

- Sí → probablemente `setosa`.
- No → realizar otra pregunta.

Después:

**¿anchura del pétalo ≤ 1.75?**

y así sucesivamente.

El modelo divide progresivamente el espacio de características hasta llegar a una predicción.

## Partes principales

### Nodo raíz

Primera pregunta del árbol.

### Nodo interno

Pregunta intermedia.

### Rama

Resultado de una condición.

### Hoja

Nodo final donde se produce una predicción.

Conceptualmente:

**datos**

↓

**pregunta**

↙ Sí  No ↘

**más preguntas**

↓

**hoja → predicción**


# 2. ¿Cómo decide qué pregunta hacer?

Un árbol prueba posibles divisiones de las variables.

Por ejemplo:

\[
x_1 \le 3.4
\]

\[
x_2 \le 1.7
\]

\[
x_3 \le 5.2
\]

y evalúa cuáles separan mejor las clases.

Para clasificación necesitamos una medida que indique cuánto están mezcladas las clases dentro de un nodo.

Una de las más utilizadas es la **impureza Gini**.


# 3. Impureza Gini

Para un nodo con clases \(1,\ldots,C\):

\[
Gini =
1 -
\sum_{c=1}^{C} p_c^2
\]

donde \(p_c\) es la proporción de observaciones de la clase \(c\).

## Nodo puro

Si todos los ejemplos pertenecen a la misma clase:

\[
p_1=1,\quad p_2=0
\]

\[
Gini = 0
\]

El nodo es completamente puro.

## Nodo mezclado

En clasificación binaria con 50 % / 50 %:

\[
Gini =
1-(0.5^2+0.5^2)
=
0.5
\]

El nodo está mucho más mezclado.

> El árbol busca divisiones que reduzcan la impureza de los nodos resultantes.


In [ ]:
def gini(proporciones):
    proporciones = np.array(proporciones)
    return 1 - np.sum(proporciones ** 2)

print("100 % / 0 %  ->", gini([1, 0]))
print("80 % / 20 % ->", gini([0.8, 0.2]))
print("50 % / 50 % ->", gini([0.5, 0.5]))


## Una división no se evalúa mirando un solo hijo

El árbol considera la impureza de los nodos resultantes ponderada por cuántas observaciones contiene cada uno.

Una división es buena si, en conjunto, produce grupos más puros que el nodo original.

No necesitamos implementar todo el algoritmo manualmente, pero sí recordar:

> **cada split intenta aumentar la separación entre las clases según el criterio elegido.**


# 4. Primer árbol real con Iris

Utilizaremos el dataset Iris para ver un árbol que podemos dibujar completo.

Empezaremos con un árbol poco profundo para que sea interpretable.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris(as_frame=True)

X_iris = iris.data
y_iris = iris.target

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_iris,
    y_iris,
    test_size=0.25,
    random_state=42,
    stratify=y_iris
)

arbol_iris = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

arbol_iris.fit(
    X_train_i,
    y_train_i
)

pred_i = arbol_iris.predict(X_test_i)

print(
    "Accuracy:",
    round(
        accuracy_score(y_test_i, pred_i),
        3
    )
)


# 5. Visualizar el árbol

En cada nodo veremos información como:

- condición utilizada;
- `gini`;
- número de muestras;
- distribución de clases;
- clase predicha.


In [ ]:
plt.figure(figsize=(18, 9))

plot_tree(
    arbol_iris,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    rounded=True,
    fontsize=9
)

plt.title("Árbol de decisión — Iris")
plt.show()


## Cómo leer un nodo

Supongamos una condición:

```text
petal length <= 2.45
```

Si es verdadera:

→ rama izquierda.

Si es falsa:

→ rama derecha.

Cada observación va recorriendo una única ruta desde la raíz hasta una hoja.

Por eso los árboles son modelos muy interpretables.


## Ver el árbol como reglas de texto


In [ ]:
reglas = export_text(
    arbol_iris,
    feature_names=list(
        iris.feature_names
    )
)

print(reglas)


# 6. Fronteras de decisión

Para entender mejor el comportamiento geométrico del árbol usaremos un dataset de dos dimensiones.

Utilizaremos `make_moons`, porque tiene una frontera no lineal y algo de ruido.


In [ ]:
from sklearn.datasets import make_moons

X_moons, y_moons = make_moons(
    n_samples=500,
    noise=0.22,
    random_state=42
)

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_moons,
    y_moons,
    test_size=0.30,
    random_state=42,
    stratify=y_moons
)

plt.figure(figsize=(8, 6))
plt.scatter(
    X_train_m[:, 0],
    X_train_m[:, 1],
    c=y_train_m,
    s=45
)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Datos de entrenamiento — dos lunas")
plt.grid(alpha=0.2)
plt.show()


## Función auxiliar para dibujar fronteras

Crearemos una rejilla de puntos, pediremos al modelo que clasifique cada posición y dibujaremos las regiones resultantes.


In [ ]:
def dibujar_frontera(
    modelo,
    X,
    y,
    titulo,
    paso=0.02
):
    x_min = X[:, 0].min() - 0.7
    x_max = X[:, 0].max() + 0.7
    y_min = X[:, 1].min() - 0.7
    y_max = X[:, 1].max() + 0.7

    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, paso),
        np.arange(y_min, y_max, paso)
    )

    rejilla = np.c_[xx.ravel(), yy.ravel()]

    pred = modelo.predict(rejilla)
    zz = pred.reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(
        xx,
        yy,
        zz,
        alpha=0.25
    )

    plt.scatter(
        X[:, 0],
        X[:, 1],
        c=y,
        s=45
    )

    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(titulo)
    plt.show()


# 7. Árbol poco profundo

Con `max_depth=2`, el árbol solo puede realizar unas pocas divisiones.

Esto limita su complejidad.


In [ ]:
tree_d2 = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

tree_d2.fit(
    X_train_m,
    y_train_m
)

dibujar_frontera(
    tree_d2,
    X_train_m,
    y_train_m,
    "Árbol de decisión — max_depth=2"
)


La frontera está formada por cortes horizontales y verticales.

Esto ocurre porque un árbol estándar divide una característica cada vez mediante condiciones del tipo:

\[
x_j \le t
\]


# 8. Árbol más profundo

Ahora permitimos un árbol mucho más complejo.


In [ ]:
tree_profundo = DecisionTreeClassifier(
    max_depth=None,
    random_state=42
)

tree_profundo.fit(
    X_train_m,
    y_train_m
)

dibujar_frontera(
    tree_profundo,
    X_train_m,
    y_train_m,
    "Árbol sin límite explícito de profundidad"
)


El árbol puede crear regiones mucho más detalladas.

Eso aumenta su capacidad para memorizar peculiaridades del train.

Pero una frontera más compleja **no implica automáticamente una mejor generalización**.


# 9. Profundidad y sobreajuste

Vamos a entrenar árboles con diferentes profundidades y comparar:

- accuracy en train;
- accuracy en test.

Si el árbol se vuelve demasiado complejo, puede seguir mejorando en train mientras deja de mejorar o empeora en test.


In [ ]:
profundidades = range(1, 21)

train_scores = []
test_scores = []

for depth in profundidades:
    modelo = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    modelo.fit(
        X_train_m,
        y_train_m
    )

    train_scores.append(
        accuracy_score(
            y_train_m,
            modelo.predict(X_train_m)
        )
    )

    test_scores.append(
        accuracy_score(
            y_test_m,
            modelo.predict(X_test_m)
        )
    )

df_depth = pd.DataFrame({
    "max_depth": list(profundidades),
    "train_accuracy": train_scores,
    "test_accuracy": test_scores
})

df_depth


In [ ]:
plt.figure(figsize=(9, 6))

plt.plot(
    df_depth["max_depth"],
    df_depth["train_accuracy"],
    marker="o",
    label="Train"
)

plt.plot(
    df_depth["max_depth"],
    df_depth["test_accuracy"],
    marker="o",
    label="Test"
)

plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Complejidad del árbol: train vs. test")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## Interpretación

Un árbol profundo tiene **baja restricción** y puede adaptarse muchísimo al entrenamiento.

Para controlar la complejidad podemos utilizar, entre otros:

- `max_depth`;
- `min_samples_split`;
- `min_samples_leaf`;
- `max_leaf_nodes`;
- `ccp_alpha`.

No necesitamos ajustar todos a la vez.

La idea es controlar cuánto puede crecer el árbol.


# 10. De un árbol a Random Forest

Un único árbol tiene dos grandes ventajas:

- es fácil de interpretar;
- captura relaciones no lineales.

Pero puede ser inestable:

> pequeños cambios en los datos pueden producir árboles diferentes.

Random Forest intenta reducir esa inestabilidad combinando muchos árboles.

## Idea central

**muchos árboles diferentes**

↓

**cada árbol predice**

↓

**se agregan las predicciones**

↓

**predicción final**

En clasificación, el resultado puede entenderse como una votación/agregación de las probabilidades de los árboles.


# 11. ¿Cómo consigue árboles diferentes?

Random Forest introduce aleatoriedad principalmente de dos formas.

## 1. Bootstrap

Cada árbol se entrena sobre una muestra del dataset obtenida **con reemplazo**.

Eso significa que:

- algunas observaciones pueden aparecer varias veces;
- otras pueden no aparecer en la muestra de un árbol concreto.

## 2. Subconjunto aleatorio de características

Al buscar una división, cada árbol no tiene por qué considerar siempre todas las variables.

Esto aumenta la diversidad entre árboles.

> Si todos los árboles fueran prácticamente idénticos, combinarlos aportaría mucho menos.


# 12. Entrenar un Random Forest


In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

forest.fit(
    X_train_m,
    y_train_m
)

pred_forest = forest.predict(
    X_test_m
)

print(
    "Accuracy test:",
    round(
        accuracy_score(
            y_test_m,
            pred_forest
        ),
        3
    )
)


`n_estimators=300` significa que el ensemble contiene 300 árboles.

No quiere decir que 300 sea siempre el valor correcto.

En general, añadir más árboles suele estabilizar el resultado, aunque aumenta el coste computacional.


# 13. Frontera de decisión del bosque

In [ ]:
dibujar_frontera(
    forest,
    X_train_m,
    y_train_m,
    "Random Forest — frontera de decisión"
)


La frontera del bosque resulta de combinar muchos árboles.

No necesitamos que cada árbol individual sea perfecto.

Queremos que el conjunto sea robusto.


# 14. Mirar algunos árboles individuales del bosque

Los árboles del Random Forest no tienen por qué tomar exactamente las mismas decisiones.

Visualizaremos la frontera de dos árboles concretos del ensemble.


In [ ]:
arbol_rf_0 = forest.estimators_[0]

dibujar_frontera(
    arbol_rf_0,
    X_train_m,
    y_train_m,
    "Árbol individual 1 dentro del Random Forest"
)


In [ ]:
arbol_rf_1 = forest.estimators_[1]

dibujar_frontera(
    arbol_rf_1,
    X_train_m,
    y_train_m,
    "Árbol individual 2 dentro del Random Forest"
)


Comparando esas fronteras con la del bosque podemos entender la idea:

> **Random Forest aprovecha la diversidad de muchos modelos individuales y agrega sus decisiones.**


# 15. Out-of-Bag (OOB)

Debido al bootstrap, cada árbol deja fuera algunas observaciones del entrenamiento.

Esas observaciones se denominan **out-of-bag** para ese árbol.

Random Forest puede utilizar esas muestras no vistas por cada árbol para obtener una estimación interna del rendimiento.

En `scikit-learn` podemos activar:

```python
oob_score=True
```

## Importante

OOB puede ser una herramienta útil, pero **no sustituye automáticamente** una estrategia completa de validación y un test independiente.


In [ ]:
forest_oob = RandomForestClassifier(
    n_estimators=400,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)

forest_oob.fit(
    X_train_m,
    y_train_m
)

print(
    "OOB score:",
    round(
        forest_oob.oob_score_,
        3
    )
)

print(
    "Accuracy test:",
    round(
        accuracy_score(
            y_test_m,
            forest_oob.predict(X_test_m)
        ),
        3
    )
)


# 16. ¿Qué ocurre al aumentar el número de árboles?

Entrenaremos bosques de diferente tamaño.

No esperamos que la accuracy aumente indefinidamente.

Normalmente llega un punto en que el rendimiento se estabiliza.


In [ ]:
n_arboles_lista = [
    1, 5, 10, 25, 50,
    100, 200, 400
]

resultados_rf = []

for n_arboles in n_arboles_lista:
    modelo = RandomForestClassifier(
        n_estimators=n_arboles,
        random_state=42,
        n_jobs=-1
    )

    modelo.fit(
        X_train_m,
        y_train_m
    )

    resultados_rf.append({
        "n_estimators": n_arboles,
        "train_accuracy": accuracy_score(
            y_train_m,
            modelo.predict(X_train_m)
        ),
        "test_accuracy": accuracy_score(
            y_test_m,
            modelo.predict(X_test_m)
        )
    })

df_rf_estimators = pd.DataFrame(
    resultados_rf
)

df_rf_estimators


In [ ]:
plt.figure(figsize=(9, 6))

plt.plot(
    df_rf_estimators["n_estimators"],
    df_rf_estimators["test_accuracy"],
    marker="o"
)

plt.xlabel("Número de árboles")
plt.ylabel("Accuracy test")
plt.title("Random Forest: efecto de n_estimators")
plt.grid(alpha=0.2)
plt.show()


# 17. Importancia de variables en modelos de árboles

Los modelos basados en árboles pueden proporcionar medidas de importancia de variables.

Una de las más habituales es la **importancia basada en reducción de impureza**.

En `scikit-learn` aparece en:

```python
model.feature_importances_
```

## Precaución

Esta medida puede estar sesgada hacia determinadas características, por ejemplo variables continuas o de alta cardinalidad.

Por eso más adelante utilizaremos también **Permutation Importance** como herramienta comparativa.


# 18. ¿Dónde entra XGBoost?

Random Forest construye muchos árboles con aleatoriedad y después los combina.

XGBoost sigue otra filosofía:

> **los árboles se construyen secuencialmente.**

Cada nueva etapa intenta mejorar el modelo acumulado respecto a la función de pérdida.

Es una implementación optimizada y regularizada de **gradient boosting**.

Conceptualmente:

**Modelo inicial**

↓

**Árbol 1 mejora la predicción**

↓

**Árbol 2 intenta corregir lo que todavía falta**

↓

**Árbol 3 vuelve a mejorar**

↓

**...**

↓

**Predicción final = combinación de muchos árboles**


# 19. Random Forest vs. Boosting

## Random Forest — Bagging

Los árboles pueden entrenarse de manera esencialmente independiente.

Después se agregan.

Objetivo intuitivo:

> reducir variabilidad y estabilizar la predicción.

---

## XGBoost — Boosting

Los árboles se añaden **secuencialmente**.

Cada nuevo árbol depende del estado actual del ensemble y de los gradientes de la función de pérdida.

Objetivo intuitivo:

> ir mejorando progresivamente el modelo acumulado.

### Simplificación útil

Para una primera intuición podemos pensar:

> “cada nueva etapa presta atención a lo que el ensemble todavía está haciendo mal”.

Pero, técnicamente, XGBoost optimiza una función objetivo mediante información de gradientes, no simplemente entrenando cada árbol sobre una lista de ejemplos fallados.


# 20. Hiperparámetros importantes de XGBoost

No necesitamos dominarlos todos.

### `n_estimators`

Número máximo de etapas/árboles de boosting.

### `learning_rate`

Controla cuánto contribuye cada nueva etapa.

Valor pequeño:

- pasos más conservadores;
- normalmente requiere más árboles.

### `max_depth`

Profundidad máxima de cada árbol.

### `subsample`

Fracción de observaciones utilizada en una iteración.

### `colsample_bytree`

Fracción de características disponible para construir cada árbol.

### `reg_alpha` y `reg_lambda`

Regularización L1 y L2.

### `tree_method="hist"`

Método eficiente basado en histogramas para construir los árboles.


# 21. XGBoost sobre el mismo dataset de dos lunas

Así podremos comparar visualmente:

- árbol;
- Random Forest;
- XGBoost.


In [ ]:
from xgboost import XGBClassifier

xgb_moons = XGBClassifier(
    n_estimators=250,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_moons.fit(
    X_train_m,
    y_train_m
)

pred_xgb_moons = xgb_moons.predict(
    X_test_m
)

print(
    "Accuracy test:",
    round(
        accuracy_score(
            y_test_m,
            pred_xgb_moons
        ),
        3
    )
)


In [ ]:
dibujar_frontera(
    xgb_moons,
    X_train_m,
    y_train_m,
    "XGBoost — frontera de decisión"
)


La frontera final sigue siendo construida mediante árboles, pero la estrategia de entrenamiento es diferente a Random Forest.


# 22. Ver cómo aprende XGBoost ronda a ronda

Una ventaja docente del boosting es que podemos observar cómo evoluciona la función de pérdida según se añaden árboles.

Separaremos temporalmente un conjunto de validación.


In [ ]:
X_train_x, X_valid_x, y_train_x, y_valid_x = train_test_split(
    X_train_m,
    y_train_m,
    test_size=0.25,
    random_state=7,
    stratify=y_train_m
)

xgb_learning = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_learning.fit(
    X_train_x,
    y_train_x,
    eval_set=[
        (X_train_x, y_train_x),
        (X_valid_x, y_valid_x)
    ],
    verbose=False
)

historial_xgb = xgb_learning.evals_result()

list(
    historial_xgb.keys()
)


## Curva de log loss

Menor log loss indica mejor ajuste según esta función de pérdida.

Observaremos train y validación.


In [ ]:
train_logloss = historial_xgb[
    "validation_0"
]["logloss"]

valid_logloss = historial_xgb[
    "validation_1"
]["logloss"]

rondas = np.arange(
    1,
    len(train_logloss) + 1
)

plt.figure(figsize=(9, 6))

plt.plot(
    rondas,
    train_logloss,
    label="Train"
)

plt.plot(
    rondas,
    valid_logloss,
    label="Validación"
)

plt.xlabel("Número de rondas de boosting")
plt.ylabel("Log loss")
plt.title("XGBoost: evolución de la pérdida")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## Qué debemos observar

Es posible que:

- la pérdida de train siga bajando;
- la de validación llegue a estabilizarse;
- o incluso empiece a empeorar.

Esto conecta nuevamente con el sobreajuste.

Por eso podemos utilizar **early stopping**.


# 23. Early stopping

La idea es sencilla:

> detener el entrenamiento si la métrica de validación deja de mejorar durante varias rondas.

Esto evita añadir árboles indefinidamente cuando ya no aportan mejora útil sobre validación.


In [ ]:
xgb_early = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    early_stopping_rounds=30,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

xgb_early.fit(
    X_train_x,
    y_train_x,
    eval_set=[
        (X_valid_x, y_valid_x)
    ],
    verbose=False
)

print(
    "Mejor iteración:",
    xgb_early.best_iteration
)

print(
    "Mejor score:",
    xgb_early.best_score
)


`best_iteration` indica la ronda con mejor resultado según el conjunto de validación utilizado para early stopping.

El conjunto utilizado para esta decisión forma parte del proceso de selección del modelo y **no debe confundirse con el test final independiente**.


# 24. Comparación sobre un dataset real

Ahora utilizaremos el dataset **Breast Cancer Wisconsin** incluido en `scikit-learn`.

Es un problema de clasificación binaria con características numéricas derivadas de imágenes de núcleos celulares.

Nuestro objetivo aquí es metodológico:

> comparar Árbol de Decisión, Random Forest y XGBoost bajo el mismo train/test.

No vamos a interpretar este ejemplo como herramienta médica real.


In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer(
    as_frame=True
)

X = cancer.data
y = cancer.target

print("Dimensiones:", X.shape)

print("\nClases:")
print(
    y.value_counts()
    .rename(
        index={
            0: cancer.target_names[0],
            1: cancer.target_names[1]
        }
    )
)

X.head()


## Separar train y test


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)


# 25. Definir los tres modelos

Utilizaremos configuraciones razonables y relativamente sencillas.

No pretendemos realizar una búsqueda exhaustiva de hiperparámetros.

La comparación sirve para estudiar comportamientos, no para declarar un ganador universal.


In [ ]:
modelos = {
    "Árbol": DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=4,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )
}

for nombre, modelo in modelos.items():
    modelo.fit(
        X_train,
        y_train
    )

print("Modelos entrenados.")


# 26. Comparar métricas

Calcularemos:

- Accuracy
- Precision
- Recall
- F1
- ROC-AUC

En un problema real deberíamos decidir qué métrica importa **antes** de escoger un modelo, según el coste de cada tipo de error.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

comparacion = []

for nombre, modelo in modelos.items():
    pred = modelo.predict(
        X_test
    )

    proba = modelo.predict_proba(
        X_test
    )[:, 1]

    comparacion.append({
        "modelo": nombre,
        "accuracy": accuracy_score(
            y_test,
            pred
        ),
        "precision": precision_score(
            y_test,
            pred
        ),
        "recall": recall_score(
            y_test,
            pred
        ),
        "f1": f1_score(
            y_test,
            pred
        ),
        "roc_auc": roc_auc_score(
            y_test,
            proba
        )
    })

df_comparacion = pd.DataFrame(
    comparacion
).set_index("modelo")

df_comparacion.round(3)


## Gráfico comparativo de métricas


In [ ]:
df_plot = (
    df_comparacion
    .reset_index()
    .melt(
        id_vars="modelo",
        var_name="metrica",
        value_name="valor"
    )
)

metricas = df_comparacion.columns

for metrica in metricas:
    datos = df_plot[
        df_plot["metrica"] == metrica
    ]

    plt.figure(figsize=(8, 5))
    plt.bar(
        datos["modelo"],
        datos["valor"]
    )
    plt.ylim(0, 1.05)
    plt.ylabel(metrica)
    plt.title(
        f"Comparación — {metrica}"
    )
    plt.grid(
        axis="y",
        alpha=0.2
    )
    plt.show()


No debemos elegir un modelo porque gane por unas centésimas en un único split.

Una comparación sólida debería considerar también:

- validación cruzada;
- incertidumbre;
- coste computacional;
- interpretabilidad;
- estabilidad;
- requisitos del problema.


# 27. Matrices de confusión

Vamos a ver exactamente qué errores comete cada modelo.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

for nombre, modelo in modelos.items():
    pred = modelo.predict(
        X_test
    )

    ConfusionMatrixDisplay.from_predictions(
        y_test,
        pred,
        display_labels=cancer.target_names
    )

    plt.title(
        f"Matriz de confusión — {nombre}"
    )
    plt.show()


# 28. Curvas ROC

La curva ROC representa:

- **True Positive Rate**;
- frente a **False Positive Rate**;

para diferentes umbrales de decisión.

El área bajo la curva es ROC-AUC.

ROC-AUC evalúa la capacidad de ordenación/separación de las clases a través de los umbrales y no depende de un único threshold de 0.5.


In [ ]:
from sklearn.metrics import RocCurveDisplay

plt.figure(figsize=(8, 6))

for nombre, modelo in modelos.items():
    RocCurveDisplay.from_estimator(
        modelo,
        X_test,
        y_test,
        name=nombre,
        ax=plt.gca()
    )

plt.title("Curvas ROC — comparación de modelos")
plt.grid(alpha=0.2)
plt.show()


# 29. Comparación con validación cruzada

Para reducir nuestra dependencia de un único train/test, podemos comparar los modelos mediante validación cruzada.

Usaremos ROC-AUC como métrica de ejemplo.


In [ ]:
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

resultados_cv = []

for nombre, modelo in modelos.items():
    scores = cross_val_score(
        modelo,
        X,
        y,
        cv=cv,
        scoring="roc_auc",
        n_jobs=1
    )

    resultados_cv.append({
        "modelo": nombre,
        "roc_auc_media": scores.mean(),
        "roc_auc_std": scores.std()
    })

df_cv = pd.DataFrame(
    resultados_cv
).set_index("modelo")

df_cv.round(3)


## Visualizar media y variabilidad


In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    df_cv.index,
    df_cv["roc_auc_media"],
    yerr=df_cv["roc_auc_std"],
    capsize=6
)

plt.ylim(0.7, 1.02)
plt.ylabel("ROC-AUC")
plt.title("ROC-AUC en validación cruzada")
plt.grid(
    axis="y",
    alpha=0.2
)
plt.show()


# 30. Importancia basada en árboles

Primero veremos la importancia interna del Random Forest.

La interpretaremos como una medida descriptiva del modelo, no como causalidad.


In [ ]:
rf_real = modelos[
    "Random Forest"
]

importancias_rf = pd.Series(
    rf_real.feature_importances_,
    index=X.columns
).sort_values(
    ascending=False
).head(12)

plt.figure(figsize=(9, 6))

importancias_rf.sort_values().plot(
    kind="barh"
)

plt.xlabel("Importancia basada en impureza")
plt.title("Random Forest — variables más importantes")
plt.show()


## Cuidado con la palabra “importante”

Que una variable sea importante para el modelo **no significa**:

- que sea causal;
- que sea clínicamente importante;
- que el modelo la utilice de forma independiente;
- que eliminarla reduzca necesariamente el rendimiento en la misma magnitud.

La importancia describe el comportamiento predictivo del modelo bajo una definición concreta.


# 31. Permutation Importance

Una forma diferente de estudiar variables es:

1. medir el rendimiento del modelo;
2. desordenar una característica;
3. volver a medir;
4. observar cuánto empeora.

Si romper una variable reduce mucho el rendimiento, el modelo dependía bastante de su información.

Utilizaremos la misma definición de importancia para comparar los tres modelos.


In [ ]:
from sklearn.inspection import permutation_importance

permutation_results = {}

for nombre, modelo in modelos.items():
    resultado = permutation_importance(
        modelo,
        X_test,
        y_test,
        n_repeats=15,
        random_state=42,
        scoring="roc_auc",
        n_jobs=-1
    )

    serie = pd.Series(
        resultado.importances_mean,
        index=X.columns
    ).sort_values(
        ascending=False
    )

    permutation_results[nombre] = serie


## Árbol — Permutation Importance

In [ ]:
top = permutation_results[
    "Árbol"
].head(10)

plt.figure(figsize=(9, 6))
top.sort_values().plot(
    kind="barh"
)
plt.xlabel(
    "Descenso medio de ROC-AUC al permutar"
)
plt.title(
    "Árbol — Permutation Importance"
)
plt.show()


## Random Forest — Permutation Importance

In [ ]:
top = permutation_results[
    "Random Forest"
].head(10)

plt.figure(figsize=(9, 6))
top.sort_values().plot(
    kind="barh"
)
plt.xlabel(
    "Descenso medio de ROC-AUC al permutar"
)
plt.title(
    "Random Forest — Permutation Importance"
)
plt.show()


## XGBoost — Permutation Importance

In [ ]:
top = permutation_results[
    "XGBoost"
].head(10)

plt.figure(figsize=(9, 6))
top.sort_values().plot(
    kind="barh"
)
plt.xlabel(
    "Descenso medio de ROC-AUC al permutar"
)
plt.title(
    "XGBoost — Permutation Importance"
)
plt.show()


Los tres modelos pueden utilizar las características de formas diferentes.

Además, cuando existen variables muy correlacionadas, la interpretación de permutation importance también debe hacerse con cuidado: una variable puede ser sustituida parcialmente por otra correlacionada.


# 32. ¿Necesitamos `StandardScaler`?

Aquí aparece una diferencia importante respecto a KNN y K-Means.

Los árboles realizan preguntas como:

\[
x_j \le t
\]

Si transformamos una variable linealmente:

\[
x'=\frac{x-\mu}{\sigma}
\]

el orden de sus observaciones no cambia.

El árbol puede simplemente elegir otro umbral equivalente.

Por eso:

> **los modelos basados en árboles normalmente no necesitan estandarización de las variables numéricas para funcionar correctamente.**

Esto simplifica bastante el preprocesamiento.


## Comprobación con un árbol

Compararemos las predicciones de un árbol entrenado:

- con variables originales;
- con variables estandarizadas.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

tree_original = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

tree_scaled = Pipeline([
    ("scaler", StandardScaler()),
    (
        "tree",
        DecisionTreeClassifier(
            max_depth=4,
            random_state=42
        )
    )
])

tree_original.fit(
    X_train,
    y_train
)

tree_scaled.fit(
    X_train,
    y_train
)

pred_original = tree_original.predict(
    X_test
)

pred_scaled = tree_scaled.predict(
    X_test
)

coincidencia = np.mean(
    pred_original == pred_scaled
)

print(
    "Proporción de predicciones coincidentes:",
    coincidencia
)


En muchos datasets serán idénticas o prácticamente idénticas.

Eso no significa que ningún tipo de transformación de datos pueda afectar jamás a un árbol.

Significa que **el escalado estándar por magnitud no es normalmente necesario**, a diferencia de métodos basados directamente en distancias.


# 33. Comparativa conceptual

| Característica | Árbol | Random Forest | XGBoost |
|---|---|---|---|
| Nº de árboles | 1 | Muchos | Muchos |
| Entrenamiento | Un árbol | Árboles con aleatoriedad | Secuencial |
| Idea principal | Reglas | Bagging / agregación | Boosting |
| Interpretabilidad | Alta | Media | Menor |
| Riesgo de sobreajuste | Alto si crece mucho | Menor que árbol individual en muchos casos | Depende mucho de regularización y tuning |
| Escalado numérico | Normalmente no necesario | Normalmente no necesario | Normalmente no necesario para booster de árboles |
| No linealidad | Sí | Sí | Sí |
| Interacciones | Sí | Sí | Sí |
| Hiperparámetros | Relativamente pocos | Más | Bastantes |
| Rendimiento tabular | Buen baseline | Muy fuerte | Frecuentemente muy competitivo |


# 34. ¿Cuál utilizar?

## Árbol de Decisión

Muy útil cuando queremos:

- explicación sencilla;
- reglas visuales;
- baseline interpretable;
- comprender el problema.

## Random Forest

Buena opción cuando queremos:

- modelo robusto;
- poco preprocesamiento;
- buen rendimiento sin tuning extremo;
- ensemble relativamente fácil de utilizar.

## XGBoost

Tiene sentido cuando:

- trabajamos con datos tabulares;
- queremos exprimir rendimiento;
- podemos dedicar tiempo a validación y tuning;
- aceptamos menor interpretabilidad directa;
- queremos controlar boosting y regularización.

> No existe un ganador universal.

El mejor modelo depende de:

- datos;
- métrica;
- tamaño del dataset;
- coste de errores;
- necesidad de explicación;
- tiempo de entrenamiento/inferencia;
- mantenimiento.


# 35. Hiperparámetros que sí merece la pena recordar

## Decision Tree

```text
max_depth
min_samples_split
min_samples_leaf
criterion
ccp_alpha
```

## Random Forest

```text
n_estimators
max_depth
min_samples_leaf
max_features
bootstrap
```

## XGBoost

```text
n_estimators
learning_rate
max_depth
subsample
colsample_bytree
reg_alpha
reg_lambda
```

No es necesario memorizar todos sus valores.

Lo importante es comprender qué dimensión de complejidad controla cada uno.


# 36. Errores frecuentes

### 1. Dejar crecer un árbol y evaluarlo solo en train

Un árbol profundo puede memorizar.

---

### 2. Pensar que Random Forest es simplemente “el mismo árbol repetido”

La diversidad entre árboles es esencial.

---

### 3. Pensar que XGBoost es un Random Forest más potente

No.

Utilizan estrategias de ensemble diferentes:

**bagging vs. boosting**.

---

### 4. Escalar por costumbre sin entender por qué

En modelos de árboles normalmente no es necesario para las variables numéricas.

---

### 5. Elegir el modelo solo por accuracy

Puede haber métricas más relevantes.

---

### 6. Utilizar el test para ajustar hiperparámetros

El test debe permanecer lo más independiente posible.

---

### 7. Interpretar feature importance como causalidad

No lo es.

---

### 8. Pensar que un modelo más sofisticado siempre debe ganar

Un árbol sencillo puede ser suficiente si el problema lo permite.


# 37. Mini reto final

Utiliza el dataset Breast Cancer de este notebook.

Sin copiar directamente la solución:

1. separa train y test;
2. entrena un `DecisionTreeClassifier`;
3. prueba tres valores de `max_depth`;
4. representa el rendimiento train/test;
5. entrena un Random Forest;
6. prueba al menos dos valores de `n_estimators`;
7. entrena XGBoost;
8. modifica `learning_rate` y `n_estimators`;
9. compara los tres modelos con:
   - accuracy;
   - precision;
   - recall;
   - F1;
   - ROC-AUC;
10. muestra las matrices de confusión;
11. representa las curvas ROC;
12. calcula permutation importance del modelo que elijas;
13. explica qué modelo seleccionarías;
14. justifica tu decisión utilizando rendimiento **y** criterios no puramente métricos.


In [ ]:
# TU CÓDIGO AQUÍ



# 38. Preguntas de comprensión

## Árboles

1. ¿Qué es un nodo raíz?
2. ¿Qué es una hoja?
3. ¿Qué representa la impureza Gini?
4. ¿Qué significa que un nodo sea puro?
5. ¿Por qué un árbol muy profundo puede sobreajustar?
6. ¿Qué controla `max_depth`?
7. ¿Por qué las fronteras de un árbol suelen tener cortes horizontales y verticales?

## Random Forest

8. ¿Por qué Random Forest utiliza muchos árboles?
9. ¿Qué es bootstrap?
10. ¿Por qué se introducen subconjuntos aleatorios de características?
11. ¿Qué significa OOB?
12. ¿Por qué añadir árboles no mejora indefinidamente el rendimiento?

## XGBoost

13. ¿Qué diferencia fundamental existe entre Random Forest y XGBoost?
14. ¿Qué significa boosting secuencial?
15. ¿Qué controla `learning_rate`?
16. ¿Qué relación suele existir entre `learning_rate` y `n_estimators`?
17. ¿Para qué sirve early stopping?
18. ¿Por qué el conjunto de early stopping no debe ser el test final?

## Comparación

19. ¿Por qué los árboles no suelen necesitar StandardScaler?
20. ¿Qué ventaja de interpretabilidad tiene un árbol frente a los ensembles?
21. ¿Qué es ROC-AUC?
22. ¿Qué información aporta una matriz de confusión?
23. ¿Por qué una importance alta no demuestra causalidad?
24. ¿Qué diferencia conceptual existe entre importance por impureza y permutation importance?
25. ¿Por qué no existe un modelo universalmente mejor?


# 39. Qué debes recordar

## Árbol de Decisión

**preguntas → divisiones → hojas → predicción**

Es intuitivo y muy interpretable, pero puede sobreajustar si crece demasiado.

---

## Random Forest

**muchos árboles + bootstrap + aleatoriedad de variables → agregación**

Busca reducir la inestabilidad de un único árbol.

---

## XGBoost

**árboles secuenciales + optimización de una función de pérdida + regularización**

Cada etapa mejora el ensemble acumulado.

---

## La comparación fundamental

### Bagging

Random Forest:

> entrenar modelos diversos y agregarlos.

### Boosting

XGBoost:

> construir modelos secuencialmente para mejorar el conjunto.

---

## Idea final

No preguntes únicamente:

> **¿Qué modelo obtiene mayor accuracy?**

Pregunta también:

- ¿generaliza?
- ¿qué errores comete?
- ¿qué métrica importa?
- ¿puedo explicarlo?
- ¿cuánto cuesta entrenarlo?
- ¿cuánto cuesta mantenerlo?
- ¿necesito realmente un modelo más complejo?

> **Un buen modelo no es necesariamente el más sofisticado: es el que resuelve mejor el problema bajo sus restricciones reales.**


# Referencias técnicas

Para preparar este notebook se han seguido las APIs y conceptos documentados oficialmente por:

- **scikit-learn**: `DecisionTreeClassifier`, `RandomForestClassifier`, árboles y ensembles.
- **XGBoost**: interfaz compatible con estimadores de scikit-learn, parámetros de tree boosting y early stopping.

El código se ha mantenido deliberadamente introductorio para uso docente.
